# [Whisper](https://arxiv.org/abs/2212.04356)

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Audio, display
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

SR = 16_000
FRAME = 0.02

processor = WhisperProcessor.from_pretrained("openai/whisper-base")

# eager attention: the fast SDPA kernels never materialize the attention matrix, so they cannot return it
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base", attn_implementation="eager"
).eval()

## Model

In [ ]:
model.model.encoder

In [ ]:
model.model.decoder

## Processor

In [ ]:
processor

In [ ]:
def load_audio(path):
    wav, _ = librosa.load(path, sr=SR)
    return wav

en_wav = load_audio("../week03/LJ001-0001.wav")
display(Audio(en_wav, rate=SR))

In [ ]:
en_wav.shape[0] / SR

In [ ]:
plt.figure(figsize=(15,1))
plt.imshow(
    processor.feature_extractor(en_wav, sampling_rate=SR).input_features[0]
)

## Tokenizer

In [ ]:
processor.tokenizer

In [ ]:
[processor.tokenizer.convert_ids_to_tokens(id) for id in processor.tokenizer.encode("hello world")]


In [ ]:
tokenizer = processor.tokenizer

tokenizer.convert_tokens_to_ids(["<|startoftranscript|>", "<|en|>", "<|ru|>", "<|transcribe|>", "<|translate|>"])

In [ ]:
SOT, EOT, SOP, NOTS = [
    tokenizer.convert_tokens_to_ids(t)
    for t in ["<|startoftranscript|>", "<|endoftext|>", "<|startofprev|>", "<|notimestamps|>"]
]
SOT, EOT, SOP, NOTS

In [ ]:
tokenizer.convert_ids_to_tokens([NOTS, NOTS + 1, NOTS + 2, len(tokenizer) - 1])

## Input and output

```
[<|startofprev|> prompt tokens]  <|startoftranscript|> <|en|> <|transcribe|> <|notimestamps|>  text ...  <|endoftext|>
```

In [ ]:
def encode(wav):
    features = processor(wav, sampling_rate=SR, return_tensors="pt").input_features
    with torch.no_grad():
        return model.model.encoder(features)

In [ ]:
encode(en_wav).last_hidden_state.shape

## Greedy decoding


In [ ]:
prefix = torch.tensor([[SOT,
                        tokenizer.convert_tokens_to_ids("<|en|>"),
                        tokenizer.convert_tokens_to_ids("<|transcribe|>"),
                        NOTS]]) # B x tokens
tokenizer.convert_ids_to_tokens(prefix[0].tolist())

In [ ]:
@torch.no_grad()
def decode(wav, language="en", task="transcribe", prompt=None, max_new_tokens=64):
    # returns: list of token ids, the prompt and the special tokens prefix included
    # your code here
    raise NotImplementedError

In [ ]:
tokenizer.decode(decode(en_wav))

## Prompting

In [ ]:
tokenizer.decode(decode(en_wav, prompt="PrinDing"))

## Alignment

In [ ]:
tokens = decode(en_wav)

with torch.no_grad():
    # your code here: out = teacher-forced forward pass over `tokens` with output_attentions=True
    raise NotImplementedError

In [ ]:
valid_enc_frames = int(en_wav.shape[0] / SR / FRAME)

In [ ]:
for layer in range(len(out.cross_attentions)):
    for head in range(model.config.decoder_attention_heads):
        print(f"layer={layer} | head={head}")
        plt.figure(figsize=(12,1))
        plt.imshow(
            out.cross_attentions[layer][0, head, :, :valid_enc_frames]
        )
        plt.show()

In [ ]:
model.generation_config.alignment_heads

## Language forcing

In [ ]:
ru_wav = load_audio("ru_new_zealand.wav")
display(Audio(ru_wav, rate=SR))

In [ ]:
# your code here: decode ru_wav with language="ru", with language="en" and with task="translate", compare the outputs